In [24]:
import sys

sys.path.append("../src")

from preprocessing import clean_test_pipeline, get_feature_columns
from training import build_error_analysis
from xgboost import Booster


In [ ]:
test = clean_test_pipeline(
    "../data/test_FD001.txt",
    "../data/RUL_FD001.txt"
)
test.shape

(13096, 27)

In [ ]:
model = Booster()

model.load_model(
    "../models/final_xgboost.json"
)

print("Model loaded successfully.")

Model loaded successfully.


In [17]:
last_cycles = (
    test.groupby("unit_number")["time_cycles"]
    .max()
    .reset_index()
)

last_cycles.head()

,unit_number,time_cycles
0,1,31
1,2,49
2,3,126
3,4,106
4,5,98


In [ ]:
test_last = (
    test.sort_values(["unit_number", "time_cycles"])
    .groupby("unit_number")
    .tail(1)
    .sort_values("unit_number")
    .reset_index(drop=True)
)

test_last.shape

(100, 27)

In [19]:
test_last[
    ["unit_number", "time_cycles", "RUL"]
].head(10)

,unit_number,time_cycles,RUL
0,1,31,112
1,2,49,98
2,3,126,69
3,4,106,82
4,5,98,91
5,6,105,93
6,7,160,91
7,8,166,95
8,9,55,111
9,10,192,96


In [20]:
feature_cols = get_feature_columns(test)

X_test = test_last[feature_cols]

X_test.shape

(100, 18)

In [21]:
import xgboost as xgb

dtest = xgb.DMatrix(X_test)

y_pred_test = model.predict(dtest)

y_pred_test[:10]

array([180.7235  , 146.04031 ,  52.25793 ,  84.483246, 101.74184 ,
       104.12921 , 103.65062 ,  75.97629 , 136.53003 , 103.48982 ],
      dtype=float32)

In [25]:
test_results=build_error_analysis(test_last,y_pred_test)
test_results.head(10)

,unit_number,time_cycles,RUL,predicted_RUL,error,absolute_error
0,1,31,112,180.723495,68.723495,68.723495
1,2,49,98,146.040314,48.040314,48.040314
2,3,126,69,52.257931,-16.742069,16.742069
3,4,106,82,84.483246,2.483246,2.483246
4,5,98,91,101.741837,10.741837,10.741837
5,6,105,93,104.129211,11.129211,11.129211
6,7,160,91,103.650620,12.650620,12.650620
7,8,166,95,75.976288,-19.023712,19.023712
8,9,55,111,136.530029,25.530029,25.530029
9,10,192,96,103.489822,7.489822,7.489822


In [23]:
from training import evaluate_model

test_metrics = evaluate_model(
    test_results["RUL"],
    test_results["predicted_RUL"],
    model_name="Final XGBoost - Test"
)

Final XGBoost - Test
MAE : 18.317594528198242
RMSE: 25.293157878153902
R²  : 0.6295355558395386


In [30]:
print("Mean error:", test_results["error"].mean())
print("MAE:", test_results["absolute_error"].mean())
print("Max error:", test_results["error"].max())
print("Min error:", test_results["error"].min())

Mean error: 10.148746418952943
MAE: 18.31759422779083
Max error: 74.72929382324219
Min error: -38.52690124511719


In [29]:
worst_test_errors = test_results.sort_values(
    "absolute_error",
    ascending=False
)

worst_test_errors.head(10)

,unit_number,time_cycles,RUL,predicted_RUL,error,absolute_error
14,15,76,83,157.729294,74.729294,74.729294
0,1,31,112,180.723495,68.723495,68.723495
66,67,71,77,142.453171,65.453171,65.453171
77,78,72,107,164.330399,57.330399,57.330399
21,22,39,111,168.168076,57.168076,57.168076
49,50,74,79,130.182968,51.182968,51.182968
1,2,49,98,146.040314,48.040314,48.040314
26,27,140,66,112.769958,46.769958,46.769958
84,85,34,118,161.523788,43.523788,43.523788
13,14,46,107,150.313431,43.313431,43.313431
